# Agent Profile Manual Test

Минимальный ноутбук для проверки user profile поверх текущего агента.

Что проверяем:
- профиль пользователя сохраняется в `data/user_profiles/`;
- агент обновляет профиль перед поиском;
- rum retrieval возвращает `top 2 FAISS + top 2 BM25` с merge duplicates;
- cocktail retrieval возвращает `top 5 BM25`;
- финальный LLM-ответ видит профиль и кандидатов, но должен выбрать максимум 2 рекомендации.

## 1. Environment And Imports

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "sommelier").exists():
    raise RuntimeError(f"Run this notebook from the repository root, got: {ROOT}")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import llm_module  # noqa: F401

from sommelier.agent.graph import (
    build_cocktail_answer_prompt,
    build_sommelier_answer_prompt,
    run_agent_turn,
)
from sommelier.agent.profile_store import load_user_profile, profile_path, save_user_profile
from sommelier.agent.state import AgentState
from sommelier.agent.tools.profile_update import profile_update

paths = {
    "product_faiss_metadata": ROOT / "data" / "indexes" / "product_faiss_metadata.json",
    "rum_search_profiles": ROOT / "data" / "catalog" / "search_profiles",
    "cocktail_search_profiles": ROOT / "data" / "catalog" / "cocktail_search_profiles",
    "user_profiles": ROOT / "data" / "user_profiles",
}

print("ROOT:", ROOT)
for name, path in paths.items():
    print(f"{name}: {path} exists={path.exists()}")

ROOT: c:\Users\egorg\Documents\SPIRT_TEST
product_faiss_metadata: c:\Users\egorg\Documents\SPIRT_TEST\data\indexes\product_faiss_metadata.json exists=True
rum_search_profiles: c:\Users\egorg\Documents\SPIRT_TEST\data\catalog\search_profiles exists=True
cocktail_search_profiles: c:\Users\egorg\Documents\SPIRT_TEST\data\catalog\cocktail_search_profiles exists=True
user_profiles: c:\Users\egorg\Documents\SPIRT_TEST\data\user_profiles exists=True


## 2. Small Display Helpers

In [2]:
def pretty_json(obj):
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump(mode="json")
    print(json.dumps(obj, ensure_ascii=False, indent=2))


def show_profile(profile):
    print("UserProfile:")
    pretty_json(profile)


def show_traces(state):
    print("Tool trace:")
    for trace in state.tool_traces:
        print(f"- {trace.tool_name}: {trace.status} | {trace.output_summary}")
        if trace.input:
            print("  input:", json.dumps(trace.input, ensure_ascii=False))


def show_rum_candidates(state):
    print("Rum candidates:")
    for index, result in enumerate(state.retrieval_results, start=1):
        profile = result.profile
        sources = state.retrieval_sources.get(result.product_id, [])
        print(f"{index}. {profile.name} [{result.product_id}] score={result.score:.3f} sources={sources}")
        print("   display:", profile.display_description)
        print("   tasting:", profile.tasting_summary)
        print("   flavors:", profile.flavor_tags)
        print("   usage:", profile.usage_tags)


def show_cocktail_candidates(state):
    print("Cocktail candidates:")
    for index, result in enumerate(state.cocktail_results, start=1):
        profile = result.profile
        print(f"{index}. {profile.name} [{result.cocktail_id}] score={result.score:.3f}")
        print("   main_rum:", profile.main_rum)
        print("   description:", profile.description)
        print("   matched_tokens:", result.matched_tokens)
        print("   ingredients:", profile.ingredients)
        print("   steps:", profile.recipe_steps[:3])

## 3. Create Or Update A Profile

Эта ячейка явно пишет предпочтения в профиль. Формулировка специально стабильная: это не одноразовый запрос, а именно вкусовые предпочтения.

In [3]:
PROFILE_SESSION_ID = "notebook-profile-demo"
PROFILE_MESSAGE = "I like vanilla and oak. I dislike coconut. I like mojito cocktails."

profile = load_user_profile(PROFILE_SESSION_ID)
updated_profile, update_result = profile_update(profile, PROFILE_MESSAGE)
profile_file = save_user_profile(updated_profile)

print("Profile update message:", PROFILE_MESSAGE)
print("Saved profile file:", profile_file)
print(update_result.summary)
print("Update metadata:")
pretty_json(update_result.metadata)
show_profile(updated_profile)

Profile update message: I like vanilla and oak. I dislike coconut. I like mojito cocktails.
Saved profile file: data\user_profiles\notebook-profile-demo.json
Profile update applied: 4 preference value(s).
Update metadata:
{
  "update": {
    "liked_flavors": [
      "vanilla",
      "oak"
    ],
    "disliked_flavors": [
      "coconut"
    ],
    "liked_cocktails": [
      "mojito"
    ],
    "disliked_cocktails": [],
    "ignored": []
  },
  "changed": {
    "liked_flavors": [
      "vanilla",
      "oak"
    ],
    "disliked_flavors": [
      "coconut"
    ],
    "liked_cocktails": [
      "mojito"
    ],
    "disliked_cocktails": []
  }
}
UserProfile:
{
  "session_id": "notebook-profile-demo",
  "liked_flavors": [
    "oak",
    "vanilla"
  ],
  "disliked_flavors": [
    "coconut",
    "sweet"
  ],
  "liked_cocktails": [
    "mojito"
  ],
  "disliked_cocktails": []
}


## 4. Inspect Saved Profile JSON

In [4]:
PROFILE_FILE = profile_path(PROFILE_SESSION_ID)
print("Profile path:", PROFILE_FILE)
print(PROFILE_FILE.read_text(encoding="utf-8"))

Profile path: data\user_profiles\notebook-profile-demo.json
{
  "session_id": "notebook-profile-demo",
  "liked_flavors": [
    "oak",
    "vanilla"
  ],
  "disliked_flavors": [
    "coconut",
    "sweet"
  ],
  "liked_cocktails": [
    "mojito"
  ],
  "disliked_cocktails": []
}


## 5. Rum Agent With Profile

Агент сам загрузит сохраненный профиль по `session_id`, обновит его при необходимости, сделает retrieval и сгенерирует ответ.

In [5]:
PROFILE_RUM_MESSAGE = "Посоветуй ром с ванилью для коктейлей"

rum_state = AgentState(
    session_id=PROFILE_SESSION_ID,
    user_message=PROFILE_RUM_MESSAGE,
    use_llm_answer=True,
)
rum_state = run_agent_turn(rum_state)

print("User message:", PROFILE_RUM_MESSAGE)
print("\nFinal answer:\n")
print(rum_state.final_answer)
print("\nParsed intent:")
pretty_json(rum_state.parsed_intent)
print("\nProfile used:")
show_profile(rum_state.user_profile)
print()
show_rum_candidates(rum_state)
print()
show_traces(rum_state)

User message: Посоветуй ром с ванилью для коктейлей

Final answer:

Конечно — для коктейлей я бы в первую очередь посмотрел на **BACARDÍ Añejo Cuatro Rum**. У него есть **мягкая ваниль, поджаренный дуб и гладкое послевкусие** — это хороший выбор, если хотите ром с более выразительным, «барным» характером для хайболов и классических миксов вроде **Cuatro Cuba Libre**.

Второй вариант — **BACARDÍ Carta Blanca**. Это **белый ром с мягкими нотами миндаля и ванили**, созданный именно для смешивания; в его профиле прямо указаны классические коктейли вроде **Mojito** и **Daiquiri**. Если вам нужен более лёгкий и универсальный ром для коктейлей, это очень удачный выбор.

Небольшая оговорка: у **Carta Blanca** и особенно у **Coquito** есть заметные кокосовые оттенки, а вы кокос не любите, поэтому я бы их оставил как запасной вариант, а не первый выбор.

Parsed intent:
{
  "intent": "search_products",
  "search": {
    "query": "Посоветуй ром с ванилью для коктейлей",
    "product_tags": [],
   

## 6. Rum Answer Prompt Preview

In [6]:
rum_prompt = build_sommelier_answer_prompt(rum_state)
print(rum_prompt[:6000])
if len(rum_prompt) > 6000:
    print("\n... truncated ...")

You are an AI sommelier assistant for rum recommendations.

Write a concise, warm recommendation in the user's language.
Use a professional but natural sommelier tone: confident, sensory, practical,
and not overblown.

Hard rules:
- Use only the product candidates and evidence provided in CONTEXT.
- Do not invent products, prices, availability, awards, ingredients, or source claims.
- If food pairing is present, say it is an inferred recommendation based on profile
  similarity, not a direct source-backed Bacardi pairing fact.
- Use user_profile as preference context, but do not invent preferences.
- Do not discard retrieved candidates solely because of profile. If a candidate
  conflicts with disliked preferences, mention the caveat or prefer another candidate.
- Prefer the best candidate first.
- Return at most 2 product recommendations.
- Explain why each recommendation matches the user request using flavor, style,
  serve, cocktail, or food-context evidence.
- Keep the answer compa

## 7. Cocktail Agent With Profile

Cocktail retrieval сейчас идет через BM25 по компактным cocktail search profiles. Профиль передается в финальный LLM-контекст.

In [7]:
PROFILE_COCKTAIL_MESSAGE = "Подбери коктейль с лаймом и мятой"

cocktail_state = AgentState(
    session_id=PROFILE_SESSION_ID,
    user_message=PROFILE_COCKTAIL_MESSAGE,
    use_llm_answer=True,
)
cocktail_state = run_agent_turn(cocktail_state)

print("User message:", PROFILE_COCKTAIL_MESSAGE)
print("\nFinal answer:\n")
print(cocktail_state.final_answer)
print("\nParsed intent:")
pretty_json(cocktail_state.parsed_intent)
print("\nNormalized cocktail query:", cocktail_state.cocktail_query)
print("\nProfile used:")
show_profile(cocktail_state.user_profile)
print()
show_cocktail_candidates(cocktail_state)
print()
show_traces(cocktail_state)

User message: Подбери коктейль с лаймом и мятой

Final answer:

Лучше всего подойдёт **Mojito** — здесь как раз есть **лайм и мята**, и это классика в твоём вкусе.

**Mojito**  
- 50 ML BACARDÍ Carta Blanca rum  
- 25 ML fresh lime juice  
- 25 ML soda  
- 15 ML sugar syrup (1:1)  
- small handful mint  
- crushed ice  

**Подача:**  
1. In a highball glass, add BACARDÍ Carta Blanca rum, fresh lime juice, sugar syrup, and soda water.  
2. Add mint leaves to the glass, gently pressing with a bar spoon to release their oils. Leave the spoon in the glass.  
3. Fill 3/4 of the glass with crushed ice. Churn the mixture with a bar spoon to combine. Top with crushed ice.  
4. Garnish with a fresh mint sprig. Serve immediately and enjoy your perfectly crafted BACARDÍ Mojito!

Если хочешь чуть более сухой и игристый вариант с тем же профилем — **Old Cuban**: mint, lime, bitters и prosecco.

Parsed intent:
{
  "intent": "cocktail_expansion",
  "search": {
    "query": "Подбери коктейль с лаймом 

## 8. Cocktail Answer Prompt Preview

In [8]:
cocktail_prompt = build_cocktail_answer_prompt(cocktail_state)
print(cocktail_prompt[:6000])
if len(cocktail_prompt) > 6000:
    print("\n... truncated ...")

You are an AI bartender-sommelier assistant for Bacardi rum cocktails.

Write a concise answer in the user's language.
Use a practical, confident, bar-service tone: helpful, sensory, and easy to follow.

Hard rules:
- Use only the cocktail candidates and evidence provided in CONTEXT.
- Do not invent cocktail names, ingredients, amounts, steps, glassware, garnish, or rum facts.
- If recipe ingredients or steps are present, preserve them exactly as provided.
- Recommend the best matching cocktail first.
- Use user_profile as preference context, but do not invent preferences.
- Prefer the best matching cocktail first.
- Return at most 2 cocktail recommendations.
- Do not mention BM25, internal search, scores, profiles, or tools.
- If the user asks for a recipe, include ingredients and preparation steps.
- If the user asks what to make with a rum or ingredient, explain why the match fits.
- Keep the answer compact and readable.

CONTEXT:
{
  "user_message": "Подбери коктейль с лаймом и мят

## 9. Food Pairing Agent With Profile

In [9]:
PROFILE_FOOD_MESSAGE = "Ем шашлык из свинины, какой ром подойдет?"

food_state = AgentState(
    session_id=PROFILE_SESSION_ID,
    user_message=PROFILE_FOOD_MESSAGE,
    use_llm_answer=True,
)
food_state = run_agent_turn(food_state)

print("User message:", PROFILE_FOOD_MESSAGE)
print("Expanded query:", food_state.expanded_query)
print("Caveat:", food_state.retrieval_caveat)
print("\nFinal answer:\n")
print(food_state.final_answer)
print()
show_rum_candidates(food_state)
print()
show_traces(food_state)

User message: Ем шашлык из свинины, какой ром подойдет?
Expanded query: Rum suitable for grilled barbecue meat. Rich, smoky, spiced rum profile. Rum with caramel, spice, oak or molasses for pork.
Caveat: Food pairing is inferred from product profile similarity and general pairing language, not from source-backed Bacardi food-pairing claims.

Final answer:

К шашлыку из свинины я бы в первую очередь посоветовал **BACARDÍ Spiced**. У него ванильная основа, древесные ноты и яркий финал с корицей и мускатным орехом — это хорошо поддержит жареное мясо и специи, особенно если подача с дымком. Если хотите пить просто и практично, он как раз подходит для смешивания, но с мясом его профиль тоже работает очень удачно.

Если хочется рома чуть более глубокого и спокойного, возьмите **BACARDÍ Reserva Ocho Rum**. В нём есть выдержанная оковая мягкость, ваниль и специи, плюс более сложный, округлый вкус — хороший вариант к свинине на мангале, когда нужен не сладкий, а более благородный акцент.

В кач

## 10. Inspect Durable Session Memory And Trace

После запусков агента выше должны появиться JSON session memory и JSONL tool traces.

In [10]:
from sommelier.agent.memory_store import load_session_memory, session_path
from sommelier.agent.trace_store import load_trace_events, trace_path

SESSION_FILE = session_path(PROFILE_SESSION_ID)
TRACE_FILE = trace_path(PROFILE_SESSION_ID)

print("Session memory file:", SESSION_FILE, "exists=", SESSION_FILE.exists())
if SESSION_FILE.exists():
    memory = load_session_memory(PROFILE_SESSION_ID)
    pretty_json(memory)

print("\nTrace JSONL file:", TRACE_FILE, "exists=", TRACE_FILE.exists())
events = load_trace_events(PROFILE_SESSION_ID)
print("Trace events:", len(events))
for event in events[-10:]:
    print(event["turn_id"], event["tool_name"], event["status"], "|", event["output_summary"])


Session memory file: data\sessions\notebook-profile-demo.json exists= True
{
  "session_id": "notebook-profile-demo",
  "messages": [
    {
      "role": "user",
      "content": "Посоветуй ром с ванилью для коктейлей",
      "timestamp": "2026-06-19T05:22:09.235979Z"
    },
    {
      "role": "assistant",
      "content": "Конечно — для коктейлей я бы в первую очередь посмотрел на **BACARDÍ Añejo Cuatro Rum**. У него есть **мягкая ваниль, поджаренный дуб и гладкое послевкусие** — это хороший выбор, если хотите ром с более выразительным, «барным» характером для хайболов и классических миксов вроде **Cuatro Cuba Libre**.\n\nВторой вариант — **BACARDÍ Carta Blanca**. Это **белый ром с мягкими нотами миндаля и ванили**, созданный именно для смешивания; в его профиле прямо указаны классические коктейли вроде **Mojito** и **Daiquiri**. Если вам нужен более лёгкий и универсальный ром для коктейлей, это очень удачный выбор.\n\nНебольшая оговорка: у **Carta Blanca** и особенно у **Coquito** е

## 11. Follow-Up Agent Check

Эта ячейка проверяет, что короткий follow-up использует предыдущий контекст из session memory.

In [11]:
FOLLOWUP_MESSAGE = "а второй вариант?"

followup_state = AgentState(
    session_id=PROFILE_SESSION_ID,
    user_message=FOLLOWUP_MESSAGE,
    use_llm_answer=True,
)
followup_state = run_agent_turn(followup_state)

print("User message:", FOLLOWUP_MESSAGE)
print("Is follow-up:", followup_state.is_followup)
print("\nEffective user message:\n")
print(followup_state.effective_user_message)
print("\nFinal answer:\n")
print(followup_state.final_answer)
print()
show_traces(followup_state)


User message: а второй вариант?
Is follow-up: True

Effective user message:

Follow-up to previous sommelier assistant turn.
Current user message: а второй вариант?
Previous user request: Ем шашлык из свинины, какой ром подойдет?
Previous search query: Rum suitable for grilled barbecue meat. Rich, smoky, spiced rum profile. Rum with caramel, spice, oak or molasses for pork.
Previous expanded query: Rum suitable for grilled barbecue meat. Rich, smoky, spiced rum profile. Rum with caramel, spice, oak or molasses for pork.
Previous recommended candidates: BACARDÍ Reserva Ocho Rum, BACARDÍ Spiced, BACARDÍ Carta Oro, BACARDÍ Carta Negra

Final answer:

Да — **второй вариант: BACARDÍ Spiced**. Он подойдёт к свиному шашлыку, если хочется более заметной пряности: здесь есть **ваниль, древесные ноты, корица и мускатный орех**, а в послевкусии — тёплая специя. Это хороший выбор, если ром нужен **для смешивания**; его профиль хорошо держится рядом с жареным мясом. *Подбор к еде здесь — по сходств

## 12. Profile Update Intent Check

Проверка именно профильного сигнала: пользователь не просит рекомендацию, а сообщает устойчивую антипатию.

In [12]:
PROFILE_DISLIKE_MESSAGE = "нет, мне не нравится сладкое"

profile_before = load_user_profile(PROFILE_SESSION_ID)
profile_after, dislike_update_result = profile_update(profile_before, PROFILE_DISLIKE_MESSAGE)
save_user_profile(profile_after)

print("Profile update message:", PROFILE_DISLIKE_MESSAGE)
print(dislike_update_result.summary)
print("Update metadata:")
pretty_json(dislike_update_result.metadata)
print("\nProfile after update:")
show_profile(profile_after)

Profile update message: нет, мне не нравится сладкое
Profile update applied: 1 preference value(s).
Update metadata:
{
  "update": {
    "liked_flavors": [],
    "disliked_flavors": [
      "sweet"
    ],
    "liked_cocktails": [],
    "disliked_cocktails": [],
    "ignored": []
  },
  "changed": {
    "liked_flavors": [],
    "disliked_flavors": [
      "sweet"
    ],
    "liked_cocktails": [],
    "disliked_cocktails": []
  }
}

Profile after update:
UserProfile:
{
  "session_id": "notebook-profile-demo",
  "liked_flavors": [
    "oak",
    "vanilla"
  ],
  "disliked_flavors": [
    "coconut",
    "sweet"
  ],
  "liked_cocktails": [
    "mojito"
  ],
  "disliked_cocktails": []
}


## 13. Optional Cleanup

Запусти эту ячейку только если хочешь удалить тестовые profile/session/trace файлы для `notebook-profile-demo`.

In [13]:
# for path in [profile_path(PROFILE_SESSION_ID), session_path(PROFILE_SESSION_ID), trace_path(PROFILE_SESSION_ID)]:
#     if path.exists():
#         path.unlink()
#         print("Deleted", path)
#     else:
#         print("No file to delete:", path)
